# E 最终 NQ75 开发实验
最终版本 r3 已按预设 smoke 规则选定。C1 复用，四组各 75 条，共 300 条输出，上限 600 次生成。
模型、证据、评分规则和推理代码保持冻结；运行完不再调参，本批不是未见收益证据。


In [ ]:
import json, os, sys, subprocess, hashlib, shutil
from pathlib import Path
import torch
assert sys.version_info >= (3, 11)
assert torch.cuda.is_available(), 'Select a GPU runtime'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
BUNDLE_NAME = 'answer_repair_final_r3_nq75_gpu'
EXPECTED_SHA256 = 'b5be903db6efecb5681498f0110ebed65920369ca1e5d466492ebae7455fc98b'
EXPECTED_CHECKSUMS_SHA256 = 'e843f615e696c1f205158744ce2ad57dc941d6b83338c716ea5711736a14383e'
OUTPUT = WORK / (BUNDLE_NAME.removesuffix('_gpu') + '_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)


In [ ]:
"""Stdlib bootstrap for intact archives and Kaggle's automatically extracted inputs.

This source is embedded in the transport notebook before the frozen code exists
on disk. The checksums document itself is pinned outside the uploaded directory.
"""
from __future__ import annotations

import hashlib
import json
import shutil
import tarfile
from pathlib import Path, PurePosixPath


def _digest(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def _verified_files(root: Path, expected_checksums_sha256: str) -> dict[str, str]:
    checksum_path = root / "bundle_checksums.json"
    if _digest(checksum_path) != expected_checksums_sha256:
        raise ValueError("Frozen checksums document hash mismatch")
    files = json.loads(checksum_path.read_bytes())
    if not isinstance(files, dict) or not files:
        raise ValueError("Invalid checksums document")
    for name, digest in files.items():
        path = PurePosixPath(name)
        if (not name or path.is_absolute() or ".." in path.parts or "\\" in name
                or not isinstance(digest, str) or len(digest) != 64):
            raise ValueError("Invalid frozen file entry")
        target = root / name
        if not target.resolve().is_relative_to(root.resolve()) or not target.is_file() or target.is_symlink():
            raise ValueError(f"Frozen file missing or unsafe: {name}")
        if _digest(target) != digest:
            raise ValueError(f"Frozen file hash mismatch: {name}")
    return files


def load_frozen_bundle(*, work: Path, input_root: Path, bundle_name: str,
                       archive_sha256: str, checksums_sha256: str) -> tuple[Path, Path | None, dict]:
    """Copy only verified payloads to a writable directory, without changing them."""
    if not bundle_name.replace("_", "").isalnum():
        raise ValueError("Unsafe bundle name")
    archive_name = bundle_name + ".tar.gz"
    local_archive = work / archive_name
    archives = ([local_archive] if local_archive.is_file() else
                sorted(input_root.rglob(archive_name)) if input_root.exists() else [])
    if len(archives) > 1:
        raise ValueError("Multiple matching archives; attach one frozen bundle")
    staging = work / ("e_v8_" + archive_sha256[:12])
    destination = staging / bundle_name
    archive = archives[0] if archives else None
    if archive is not None:
        if _digest(archive) != archive_sha256:
            raise ValueError("Archive hash mismatch")
        if not destination.exists():
            staging.mkdir(parents=True, exist_ok=True)
            with tarfile.open(archive) as handle:
                handle.extractall(staging, filter="data")
        files = _verified_files(destination, checksums_sha256)
        mode = "intact_archive"
        source = str(archive)
    else:
        roots = ([p.parent for p in input_root.rglob("bundle_checksums.json")
                  if p.parent.name == bundle_name] if input_root.exists() else [])
        if len(roots) != 1:
            raise ValueError(f"Expected one archive or auto-extracted bundle, found {len(roots)} extracted roots")
        source_root = roots[0]
        files = _verified_files(source_root, checksums_sha256)
        destination.mkdir(parents=True, exist_ok=True)
        # Ignore unverified extra files rather than adding them to Python's import path.
        for name in [*files, "bundle_checksums.json"]:
            source_file, target = source_root / name, destination / name
            if target.exists():
                if _digest(target) != _digest(source_file):
                    raise ValueError(f"Existing writable bundle differs: {name}")
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(source_file, target)
        _verified_files(destination, checksums_sha256)
        mode, source = "kaggle_auto_extracted", str(source_root)
    report = {"transport": mode, "source": source, "bundle": str(destination),
              "expected_archive_sha256": archive_sha256,
              "archive_hash_verified": archive is not None,
              "checksums_sha256": checksums_sha256, "payload_hash_checked_files": len(files),
              "algorithm_or_input_modified": False}
    return destination, archive, report

if not Path('/kaggle/input').exists() and not (WORK / (BUNDLE_NAME + '.tar.gz')).exists():
    from google.colab import files
    uploaded = files.upload()
    upload_name = BUNDLE_NAME + '.tar.gz'
    assert upload_name in uploaded
    (WORK / upload_name).write_bytes(uploaded[upload_name])
BUNDLE, archive, bootstrap = load_frozen_bundle(
    work=WORK, input_root=Path('/kaggle/input'), bundle_name=BUNDLE_NAME,
    archive_sha256=EXPECTED_SHA256, checksums_sha256=EXPECTED_CHECKSUMS_SHA256)
(OUTPUT / 'bootstrap.json').write_text(json.dumps(bootstrap, indent=2) + '\n')
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
assert manifest['version'] == 'answer_repair_v8r3_nq75_final_v1' and manifest['samples'] == 75
for name in ('manifest.json', 'bundle_checksums.json'):
    shutil.copy2(BUNDLE / name, OUTPUT / name)
print('Verified bundle:', bootstrap, manifest['review_numbers'])


In [ ]:
def run_logged(argv, name):
    with (OUTPUT / name).open('a', encoding='utf-8') as log:
        log.write('COMMAND ' + json.dumps(argv) + '\n'); log.flush()
        process = subprocess.Popen(argv, cwd=BUNDLE, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if code: raise subprocess.CalledProcessError(code, argv)

run_logged([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-answer-repair-v8.txt'], 'install.log')
run_logged([sys.executable, '-m', 'pip', 'freeze'], 'packages.log')
run_logged([sys.executable, 'scripts/check_answer_repair_environment.py', '--require-cuda',
            '--out', str(OUTPUT / 'environment.json')], 'environment.log')
adapter_source = '"""Run frozen tests with Python 3.11\'s AST schema in this TEST process only.\n\nPython 3.12 added an empty type_params field to ordinary function/class ASTs;\nPython 3.13 changed dumping of empty/optional fields. Serialize all legacy\nfields explicitly and omit only the new empty type_params field.\nNonempty type parameters are rejected rather than silently discarded. Nothing\nin the frozen bundle or any model-inference process is patched or rewritten.\n"""\nfrom __future__ import annotations\n\nimport ast\nimport sys\nimport unittest\nfrom pathlib import Path\n\ndef canonical_dump(node, annotate_fields=True, include_attributes=False, *, indent=None):\n    if not isinstance(node, ast.AST) or indent is not None:\n        raise AssertionError("The frozen fingerprint uses an unindented AST dump")\n\n    def render(value):\n        if isinstance(value, ast.AST):\n            fields = []; keywords = annotate_fields\n            for name in value._fields:\n                try:\n                    child = getattr(value, name)\n                except AttributeError:\n                    keywords = True\n                    continue\n                if name == "type_params" and isinstance(value, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):\n                    if child:\n                        raise AssertionError("Nonempty type parameters cannot be compared using the frozen Python 3.11 AST schema")\n                    continue\n                if child is None and getattr(type(value), name, ...) is None:\n                    keywords = True\n                    continue\n                fields.append((name + "=" if keywords else "") + render(child))\n            if include_attributes:\n                fields.extend(name + "=" + render(getattr(value, name)) for name in value._attributes\n                              if hasattr(value, name) and not (getattr(value, name) is None\n                                  and getattr(type(value), name, ...) is None))\n            return type(value).__name__ + "(" + ", ".join(fields) + ")"\n        if isinstance(value, list):\n            return "[" + ", ".join(render(item) for item in value) + "]"\n        return repr(value)\n\n    return render(node)\n\n\ndef main():\n    # The notebook launches this external adapter with cwd set to the verified\n    # frozen bundle. Make that bundle\'s tests/imports take precedence.\n    if not (Path.cwd() / "bundle_checksums.json").is_file():\n        raise AssertionError("Run the test adapter from the verified frozen bundle")\n    sys.path.insert(0, str(Path.cwd()))\n    ast.dump = canonical_dump\n    print("Frozen preflight AST schema: Python 3.11; only empty type_params omitted; test process only.", flush=True)\n    unittest.main(module=None, argv=sys.argv)\n\n\nif __name__ == "__main__":\n    main()\n'
adapter_path = OUTPUT / 'preflight_ast_compat.py'
adapter_path.write_text(adapter_source)
assert hashlib.sha256(adapter_path.read_bytes()).hexdigest() == 'dac5b7ec6c3c176f3503df32d601de937bc1ed0a5c3c2c4dd251589a208e959d'
(OUTPUT / 'preflight_adapter.json').write_text(json.dumps({
    'adapter_sha256': 'dac5b7ec6c3c176f3503df32d601de937bc1ed0a5c3c2c4dd251589a208e959d', 'frozen_bundle_sha256': EXPECTED_SHA256,
    'scope': 'unittest process only; no inference changes'
}, indent=2) + '\n')
run_logged([sys.executable, str(adapter_path), *manifest['portable_tests']], 'tests.log')
RUNS = []
for arm in manifest['protocol']:
    strategy = arm['strategy']
    argv = [sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', 'nq',
        '--c1', str(BUNDLE / manifest['scored_input']), '--all', '--strategy', strategy,
        '--max-new-tokens', str(arm['max_new_tokens']), '--out', str(OUTPUT / (strategy + '.jsonl'))]
    run_logged(argv + ['--check-only'], strategy + '_preflight.log')
    RUNS.append((strategy, argv))


In [ ]:
try:
    for strategy, argv in RUNS:
        run_logged(argv, strategy + '_run.log')
finally:
    partial = shutil.make_archive(str(WORK / (OUTPUT.name + '_partial')), 'zip', OUTPUT)
    print('Recovery archive:', partial)


In [ ]:
run_logged([sys.executable, 'scripts/summarize_answer_repair_v8.py',
            '--bundle', str(BUNDLE), '--results', str(OUTPUT)], 'smoke_check.log')
report = json.loads((OUTPUT / 'comparison.json').read_text())
export = shutil.make_archive(str(WORK / (OUTPUT.name + '_export')), 'zip', OUTPUT)
print('Result ZIP:', export)
print('SHA256:', hashlib.sha256(Path(export).read_bytes()).hexdigest())
print('All arms complete:', report['all_arms_complete'])
print('Final version remains r3; no post-run selection.')
print('Human output review pending; inference version remains frozen.')
if not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(export)
